<a href="https://colab.research.google.com/github/elsheppardo/hello-world/blob/main/Stage_6_Summary_2022_2023_Crypto_Gains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STAGE 6: Final Summary — 2022 and 2023 Capital Gains
======================================================
Purpose: Executive summary dashboard that consolidates the headline
         numbers from Stages 4 and 5 into a single sheet. Intended
         as the top-page reference for the accountant.

Output: Stage6_Summary.xlsx

Contents:
  Section 1: 2022 Summary
    - Breakdown by event type (AVAX, BTC trading, Anchor/aUST)
    - Net capital loss
    - Taxable (allowable loss) at 50% inclusion

  Section 2: 2023 Summary
    - BTC trading subtotal
    - USD→CAD FX gain subtotal
    - Stock margin account gain (user input)
    - Net capital gain
    - Taxable gain at 50% inclusion

  Section 3: Two-Year Consolidated
    - Loss carry-forward application
    - Net taxable capital gains
    - Framework for RRSP deduction impact

  Section 4: Flagged Items for Accountant Review
    - Methodology choices
    - Assumed values that need verification
    - Items pending source documentation

Note: All numbers are hardcoded from Stages 4 and 5 for this standalone
summary. For drilldown detail, refer to the individual stage files.
"""

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT_NAME = "Arial"
styles = {
    'title':       Font(name=FONT_NAME, size=18, bold=True, color='1F4E79'),
    'section':     Font(name=FONT_NAME, size=14, bold=True, color='1F4E79'),
    'subsection':  Font(name=FONT_NAME, size=12, bold=True, color='2E75B6'),
    'header':      Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
    'body':        Font(name=FONT_NAME, size=10),
    'body_bold':   Font(name=FONT_NAME, size=10, bold=True),
    'input':       Font(name=FONT_NAME, size=10, color='0000FF'),
    'formula':     Font(name=FONT_NAME, size=10, color='000000'),
    'note':        Font(name=FONT_NAME, size=9,  italic=True, color='595959'),
    'key_result':  Font(name=FONT_NAME, size=12, bold=True, color='006100'),
    'loss_result': Font(name=FONT_NAME, size=12, bold=True, color='9C0006'),
    'header_fill':    PatternFill('solid', fgColor='1F4E79'),
    'subheader_fill': PatternFill('solid', fgColor='D5E8F0'),
    'input_fill':     PatternFill('solid', fgColor='FFF2CC'),
    'total_fill':     PatternFill('solid', fgColor='E2EFDA'),
    'warning_fill':   PatternFill('solid', fgColor='FCE4D6'),
    'key_fill':       PatternFill('solid', fgColor='C6EFCE'),
    'loss_fill':      PatternFill('solid', fgColor='FFC7CE'),
}
thin = Side(border_style='thin', color='BFBFBF')
cell_border = Border(left=thin, right=thin, top=thin, bottom=thin)

FMT_CAD = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_PCT = '0.0%'

# ============================================================
# Key numbers from Stages 4 and 5 (hardcoded for standalone summary)
# ============================================================

# 2022 (from Stage 4)
NUM_2022_AVAX_LOSS       = -33.04
NUM_2022_BTC_TO_UST_GAIN = 9920.18
NUM_2022_AUST_LOSS       = -21939.25
NUM_2022_JUL1_BTC_GAIN   = 1712.54
NUM_2022_JUL14_BTC_GAIN  = 1639.95
NUM_2022_NET_CAP         = NUM_2022_AVAX_LOSS + NUM_2022_BTC_TO_UST_GAIN + NUM_2022_AUST_LOSS + NUM_2022_JUL1_BTC_GAIN + NUM_2022_JUL14_BTC_GAIN
# = -8,699.62

# 2023 BTC trading (from Stage 5)
NUM_2023_BTC_JAN14_LOSS  = -4502.97
NUM_2023_BTC_MAR22_LOSS  = -1534.50
NUM_2023_BTC_NOV30_GAIN  = 694.81
NUM_2023_BTC_NET         = NUM_2023_BTC_JAN14_LOSS + NUM_2023_BTC_MAR22_LOSS + NUM_2023_BTC_NOV30_GAIN
# = -5,342.66

# 2023 FX gain on USD→CAD conversions (from Stage 5)
NUM_2023_FX_GAIN         = 32256.13

# 2023 crypto total
NUM_2023_CRYPTO_NET      = NUM_2023_BTC_NET + NUM_2023_FX_GAIN
# = 26,913.48

# 2023 stock margin gain (user input)
NUM_2023_STOCK_GAIN      = 5000.00

# 2023 grand total capital gains
NUM_2023_GROSS_CAP       = NUM_2023_CRYPTO_NET + NUM_2023_STOCK_GAIN
# = 31,913.48


def build_workbook(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Summary"

    set_col_widths(ws, [4, 58, 18, 18, 40])

    # ============================================================
    # Title
    # ============================================================
    ws.cell(row=1, column=2, value="Capital Gains Summary — 2022 and 2023").font = styles['title']
    ws.cell(row=2, column=2, value="Executive summary for accountant review. Underlying detail in Stages 1-5.").font = styles['note']

    r = 4

    # ============================================================
    # SECTION 1: 2022 Summary
    # ============================================================
    c = ws.cell(row=r, column=2, value="2022 Tax Year — Capital Gains"); c.font = styles['section']
    r += 2

    header_row(ws, r, ["Event", "CAD Amount", "Notes"], col_start=2)
    r += 1

    items_2022 = [
        ("AVAX/LUNA disposal (Feb 24 bridge to Terra)",   NUM_2022_AVAX_LOSS,       "At 2021 carry-forward ACB of $33,950 CAD"),
        ("BTC → UST swap (March 2022 aggregate)",          NUM_2022_BTC_TO_UST_GAIN, "7.31 BTC disposed at KuCoin; small gain on FX/price"),
        ("aUST disposition (May 10, 2022 Terra escape)",   NUM_2022_AUST_LOSS,       "Single capital event; includes Anchor yield under value-accruing token treatment"),
        ("Paxos BTC sell (Jul 1, 2022)",                   NUM_2022_JUL1_BTC_GAIN,   "0.98 BTC"),
        ("Paxos BTC sell (Jul 14, 2022)",                  NUM_2022_JUL14_BTC_GAIN,  "1.00 BTC"),
    ]
    start_2022 = r
    for event, amount, note in items_2022:
        ws.cell(row=r, column=2, value=event).font = styles['body']
        c = ws.cell(row=r, column=3, value=amount); c.font = styles['input']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=5, value=note); c.font = styles['note']
        for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
        r += 1
    end_2022 = r - 1

    # Subtotal row
    c = ws.cell(row=r, column=2, value="2022 Net Capital Gain/(Loss)"); c.font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=SUM(C{start_2022}:C{end_2022})")
    c.font = styles['loss_result']; c.fill = styles['loss_fill']; c.number_format = FMT_CAD
    row_2022_net = r
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['loss_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 1

    # 50% row
    c = ws.cell(row=r, column=2, value="2022 Allowable Capital Loss (50%)"); c.font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=C{row_2022_net}*0.5")
    c.font = styles['loss_result']; c.fill = styles['loss_fill']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=5, value="Can be carried back 3 years or forward indefinitely, OR applied against 2023 gains")
    c.font = styles['note']; c.fill = styles['loss_fill']
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['loss_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 3

    # ============================================================
    # SECTION 2: 2023 Summary
    # ============================================================
    c = ws.cell(row=r, column=2, value="2023 Tax Year — Capital Gains"); c.font = styles['section']
    r += 2

    # BTC sub-summary
    c = ws.cell(row=r, column=2, value="BTC Trading (2023)"); c.font = styles['subsection']
    r += 1
    header_row(ws, r, ["Event", "CAD Amount", "Notes"], col_start=2)
    r += 1

    items_2023_btc = [
        ("Paxos BTC sell (Jan 14, 2023) — 2 BTC",      NUM_2023_BTC_JAN14_LOSS, "Sold below ACB"),
        ("KuCoin BTC sell (Mar 22, 2023) — 2.04 BTC",  NUM_2023_BTC_MAR22_LOSS, "Main final disposition"),
        ("KuCoin BTC residual (Nov 30, 2023)",          NUM_2023_BTC_NOV30_GAIN, "Small residual, estimated"),
    ]
    start_btc = r
    for event, amount, note in items_2023_btc:
        ws.cell(row=r, column=2, value=event).font = styles['body']
        c = ws.cell(row=r, column=3, value=amount); c.font = styles['input']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=5, value=note); c.font = styles['note']
        for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
        r += 1
    end_btc = r - 1

    c = ws.cell(row=r, column=2, value="  BTC Trading Subtotal"); c.font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=SUM(C{start_btc}:C{end_btc})")
    c.font = styles['formula']; c.number_format = FMT_CAD
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['total_fill']
        ws.cell(row=r, column=col).border = cell_border
    row_btc_subtotal = r
    r += 2

    # FX gain sub-summary
    c = ws.cell(row=r, column=2, value="USD→CAD Conversions (FX Gain on USD Pool)"); c.font = styles['subsection']
    r += 1
    ws.cell(row=r, column=2, value="8 NDAX EFT withdrawals Feb–Dec 2023").font = styles['body']
    c = ws.cell(row=r, column=3, value=NUM_2023_FX_GAIN); c.font = styles['input']; c.number_format = FMT_CAD
    ws.cell(row=r, column=5, value="ACB per USD = 1.27268 (Jan/Mar 2022 purchases); cashed out at ~1.35-1.38").font = styles['note']
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    row_fx_gain = r
    r += 2

    # Stock margin
    c = ws.cell(row=r, column=2, value="Stock Margin Account Gain (2023)"); c.font = styles['subsection']
    r += 1
    ws.cell(row=r, column=2, value="User-reported gain from margin account").font = styles['body']
    c = ws.cell(row=r, column=3, value=NUM_2023_STOCK_GAIN); c.font = styles['input']; c.fill = styles['input_fill']; c.number_format = FMT_CAD
    ws.cell(row=r, column=5, value="VERIFY exact amount with your brokerage statements").font = styles['note']
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    row_stock = r
    r += 2

    # 2023 total
    c = ws.cell(row=r, column=2, value="2023 Gross Net Capital Gain"); c.font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=C{row_btc_subtotal}+C{row_fx_gain}+C{row_stock}")
    c.font = styles['key_result']; c.fill = styles['key_fill']; c.number_format = FMT_CAD
    row_2023_gross = r
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['key_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 3

    # ============================================================
    # SECTION 3: Two-Year Consolidated
    # ============================================================
    c = ws.cell(row=r, column=2, value="Two-Year Consolidated — Applying 2022 Loss to 2023 Gain"); c.font = styles['section']
    r += 2

    ws.cell(row=r, column=2, value="2023 Gross Capital Gains").font = styles['body']
    c = ws.cell(row=r, column=3, value=f"=C{row_2023_gross}"); c.font = styles['formula']; c.number_format = FMT_CAD
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    r += 1

    ws.cell(row=r, column=2, value="Less: 2022 Net Capital Loss Carryback/Forward").font = styles['body']
    c = ws.cell(row=r, column=3, value=f"=C{row_2022_net}"); c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=5, value="Applied against 2023 gain").font = styles['note']
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    row_carryback = r
    r += 1

    ws.cell(row=r, column=2, value="Net Capital Gain (after loss application)").font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=C{row_2023_gross}+C{row_carryback}"); c.font = styles['body_bold']; c.number_format = FMT_CAD
    row_net_after_loss = r
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['total_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 1

    ws.cell(row=r, column=2, value="Taxable Capital Gain (50% inclusion)").font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=C{row_net_after_loss}*0.5"); c.font = styles['key_result']; c.fill = styles['key_fill']; c.number_format = FMT_CAD
    ws.cell(row=r, column=5, value="This amount enters 2023 taxable income").font = styles['note']
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['key_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 3

    # ============================================================
    # SECTION 4: Tax context / framework
    # ============================================================
    c = ws.cell(row=r, column=2, value="2023 Tax Context (rough framework — accountant to confirm)"); c.font = styles['section']
    r += 2

    ws.cell(row=r, column=2, value="Employment income (user-provided)").font = styles['body']
    c = ws.cell(row=r, column=3, value=155000); c.font = styles['input']; c.fill = styles['input_fill']; c.number_format = FMT_CAD
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    row_employment = r
    r += 1

    ws.cell(row=r, column=2, value="Plus: Taxable capital gain (from above)").font = styles['body']
    c = ws.cell(row=r, column=3, value=f"=C{row_net_after_loss}*0.5"); c.font = styles['formula']; c.number_format = FMT_CAD
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    r += 1

    ws.cell(row=r, column=2, value="Less: RRSP deduction (user-provided)").font = styles['body']
    c = ws.cell(row=r, column=3, value=-65000); c.font = styles['input']; c.fill = styles['input_fill']; c.number_format = FMT_CAD
    ws.cell(row=r, column=5, value="Can be deducted in 2023 or carried forward — accountant to optimize").font = styles['note']
    for col in range(2, 6): ws.cell(row=r, column=col).border = cell_border
    row_rrsp = r
    r += 1

    ws.cell(row=r, column=2, value="Estimated 2023 Taxable Income").font = styles['body_bold']
    c = ws.cell(row=r, column=3, value=f"=C{row_employment}+C{row_net_after_loss}*0.5+C{row_rrsp}")
    c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_CAD
    row_taxable_income = r
    for col in range(2, 6):
        ws.cell(row=r, column=col).fill = styles['total_fill']
        ws.cell(row=r, column=col).border = cell_border
    r += 3

    # ============================================================
    # SECTION 5: Flagged items for accountant
    # ============================================================
    c = ws.cell(row=r, column=2, value="Items Flagged for Accountant Review"); c.font = styles['section']
    r += 2

    flagged_items = [
        ("Methodology choices", [
            "• Capital gains treatment for Anchor aUST (rather than interest income): Based on aUST being a "
            "value-accruing receipt token. Not universally agreed; firm should confirm position.",
            "• USD stablecoins (USDT, USDC, USD cash) pooled with single CAD cost basis: Practical and "
            "defensible; conservative alternative is to track each separately.",
            "• BTC ACB pooled across Paxos and KuCoin (CRA-required for identical property).",
            "• Paxos wire reconciliation: $286,845 USD purchased vs $286,815 wired ($30 wire fee, immaterial).",
        ]),
        ("Assumed values requiring verification", [
            "• UST depegged swap rate on May 10, 2022: assumed $0.90/UST. Actual rate between $0.80-$0.95 "
            "during dead-cat bounce. Each $0.10 shift = ~$14k CAD impact on 2022 loss.",
            "• AVAX price on Feb 24, 2022: assumed $76 USD. Verify against CoinGecko/CoinMarketCap archive.",
            "• FX rates throughout: estimates; verify against Bank of Canada FXUSDCAD daily (Valet API).",
            "• BTC price on Nov 30, 2023 (residual cleanup): estimated $37,230 USD.",
        ]),
        ("Items with incomplete documentation", [
            "• Feb 24, 2022 unexplained UST deposit to Terra (13,453.64 UST): Source not identified in "
            "available CSVs. ACB assumed at ~$1.00 × FX. If source is found, adjust ACB accordingly.",
            "• Feb 26, 2023 NDAX deposit ($65,633 USDT): Tagged as from KuCoin but missing from KuCoin "
            "withdrawal CSV (CSV has gap Nov 2022 – Mar 2023). Treated as consistent with other KC→NDAX transfers.",
            "• Avalanche chain transaction CSV: not available. 2021 end-of-year valuation ($33,950 CAD from "
            "2021 summary) carried forward as AVAX ACB.",
            "• Nov 2022 USDT transfers (~$34,048) to Paxos-linked wallet: Not reflected as Paxos deposits; "
            "treated as internal USD pool transfer.",
        ]),
        ("Tax planning notes (accountant to decide/optimize)", [
            "• 2022 capital loss application: Against 2023 gains (shown here) or carryback to 2019/2020/2021 "
            "if those years had gains. Can also carry forward indefinitely.",
            "• RRSP deduction timing: Can be used in 2022, 2023, or future years. Strategic choice depending on "
            "marginal rates in each year.",
            "• Filing status: Both 2022 and 2023 returns are late. 2022 nets to a loss (no penalty risk). "
            "2023 expected refund based on employment tax withholding + RRSP deduction (no penalty risk on refund).",
        ]),
    ]

    for title, items in flagged_items:
        c = ws.cell(row=r, column=2, value=title); c.font = styles['subsection']
        r += 1
        for item in items:
            c = ws.cell(row=r, column=2, value=item); c.font = styles['note']
            c.alignment = Alignment(horizontal='left', vertical='top', wrap_text=True)
            ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=5)
            ws.row_dimensions[r].height = 35
            r += 1
        r += 1

    # ============================================================
    # SECTION 6: Companion documents
    # ============================================================
    c = ws.cell(row=r, column=2, value="Companion Files"); c.font = styles['section']
    r += 2
    companion_files = [
        "• Crypto_Trading_History_Reconstruction.docx — full narrative of platform-by-platform reconstruction",
        "• Stage1_FX_BTC_Rates.xlsx — historical FX and BTC price lookup table",
        "• Stage2_Fiat_Cost_Basis.xlsx — CAD→USD purchase records, establishes ACB per USD (1.27268)",
        "• Stage3_BTC_ACB_Ledger.xlsx — running BTC ACB across Paxos and KuCoin",
        "• Stage4_2022_Events.xlsx — all 2022 disposition events with CAD gain/loss",
        "• Stage5_2023_Events.xlsx — all 2023 disposition events including FX conversions",
        "• Original CSV exports (Paxos, KuCoin, Terra, NDAX) — source transaction data",
    ]
    for line in companion_files:
        c = ws.cell(row=r, column=2, value=line); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        r += 1

    r += 1
    c = ws.cell(row=r, column=2, value=(
        "Prepared as accountant-companion material. All methodology, assumptions, and final tax positions "
        "should be reviewed and confirmed by a qualified Canadian tax professional."
    )); c.font = styles['note']

    wb.save(output_path)
    print(f"Saved: {output_path}")


def set_col_widths(ws, widths):
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w


def header_row(ws, row, headers, col_start=1):
    for i, h in enumerate(headers):
        cell = ws.cell(row=row, column=col_start + i, value=h)
        cell.font = styles['header']
        cell.fill = styles['header_fill']
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = cell_border


if __name__ == "__main__":
    build_workbook("Stage6_Summary.xlsx")